# 02 - Preparar dataset limpio

Objetivo: balancear clases y dividir imagenes en `train`, `validation` y `test`.

In [ ]:
from pathlib import Path
import random
import shutil

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SOURCE_DIR = PROJECT_ROOT / 'dataset_frames'
OUTPUT_DIR = PROJECT_ROOT / 'dataset_clean'
CLASSES = ['arbejas', 'arroz', 'frijol', 'maiz_pira']
RANDOM_SEED = 42
SPLITS = {'train': 0.70, 'validation': 0.20, 'test': 0.10}

random.seed(RANDOM_SEED)

In [ ]:
def prepare_clean_dataset():
    if OUTPUT_DIR.exists():
        shutil.rmtree(OUTPUT_DIR)

    class_images = {}
    for cls in CLASSES:
        images = sorted((SOURCE_DIR / cls).glob('*.jpg'))
        random.shuffle(images)
        class_images[cls] = images

    min_count = min(len(images) for images in class_images.values())
    summary = {}

    for cls, images in class_images.items():
        selected = images[:min_count]
        train_end = int(min_count * SPLITS['train'])
        val_end = train_end + int(min_count * SPLITS['validation'])
        split_map = {
            'train': selected[:train_end],
            'validation': selected[train_end:val_end],
            'test': selected[val_end:]
        }

        for split, split_images in split_map.items():
            target_dir = OUTPUT_DIR / split / cls
            target_dir.mkdir(parents=True, exist_ok=True)
            for image in split_images:
                shutil.copy2(image, target_dir / image.name)
            summary[(split, cls)] = len(split_images)

    return summary

# prepare_clean_dataset()